# Notebook: Carga de Listas Riesgo y Cumplimiento + Reportes de Scoring

Este notebook:
1. Carga 5 archivos CSV desde ADLS (vía BigQuery Omni - External Tables) hacia tablas nativas de `raw_dataentry_riesgos` (truncate + insert con campos de auditoría).
2. Ejecuta los 2 reportes de Scoring de Riesgo (Persona Natural / Persona Jurídica) y los exporta como CSV a GCS.

**Diseño**: parametrizable, modular, configurable y desacoplado — toda la lógica de carga de archivos está controlada por un diccionario de configuración (`ARCHIVOS_CONFIG`). Agregar una nueva lista no requiere tocar la lógica, solo agregar una entrada.

# 1. Librerías e inicialización de clientes

In [ ]:
from datetime import datetime
import pytz
import io
import gzip
from google.cloud import storage
from google.cloud import bigquery

client = storage.Client()
clientBQ = bigquery.Client()


# 2. Variables Globales / Configuración
Todo lo que pueda cambiar entre entornos (dev/qa/prd) o futuras listas vive aquí. **No se debe modificar código fuera de esta sección para operar el notebook en el día a día.**

In [ ]:
# ---------------------------------------------------------------------------
# Zona horaria / fecha de proceso
# ---------------------------------------------------------------------------
ZONA = pytz.timezone('America/Lima')
PERU_TIME = datetime.now(ZONA)

VAR_PROCESS_DATE   = PERU_TIME.strftime('%Y-%m-%d')          # process_date (DATE)
VAR_LOAD_DATE      = PERU_TIME.strftime('%Y-%m-%d %H:%M:%S') # load_date (DATETIME)
VAR_FECHA_ARCHIVO  = PERU_TIME.strftime('%Y%m%d')
VAR_ANHO           = PERU_TIME.strftime('%Y')
VAR_MES            = PERU_TIME.strftime('%m')

# ---------------------------------------------------------------------------
# Proyectos / Dataset / Conexión Omni (mismos que el notebook de referencia)
# ---------------------------------------------------------------------------
#@title Variables Generales { run: "auto", display-mode: "form" }
var_project_storage   = "prd-izipay-data-storage-pv" #@param {type:"string"}
var_project_operation = "prd-izipay-data-operation"  #@param {type:"string"}
var_dataset_destino   = "raw_dataentry_riesgos"      #@param {type:"string"}   # dataset destino (tablas nativas)
var_dataset_stage     = "bq_omni_izipay_azure_saizipaydatamarts" #@param {type:"string"}  # dataset Omni (region azure-eastus2) - SOLO external tables
var_dataset_tmp       = "raw_stage_dataentry_riesgos"            #@param {type:"string"}  # dataset estandar BQ (region US) - aqui se materializa la temporal
var_connection        = "azure-eastus2.bq-omni-izipay-azure-saizipaydatamarts" #@param {type:"string"}

# ---------------------------------------------------------------------------
# Parametros de formato CSV (igual que el patron usado para Comercios 3DS)
# ---------------------------------------------------------------------------
var_field_delimiter   = ","  #@param {type:"string"}   # cambiar a ';' si el plano viene con punto y coma
var_quote             = '"'  #@param {type:"string"}   # caracter de comillas usado en el CSV

# ---------------------------------------------------------------------------
# Campos de auditoría por defecto (constantes de negocio)
# ---------------------------------------------------------------------------
RECORD_SOURCE  = 'Data Entry Riesgo'
CREATION_USER  = 'MC2253'

# ---------------------------------------------------------------------------
# Ruta base en ADLS donde están los planos CSV
# ---------------------------------------------------------------------------
RUTA_BASE_ADLS = "azure://saizipaydatamarts.blob.core.windows.net/adls-ingesta/DataEntry/Riesgo Opertativo y Cumplimiento/Scoring"

# ---------------------------------------------------------------------------
# Bucket / ruta de salida de reportes en GCS
# ---------------------------------------------------------------------------
bucket_name        = "adls-reportes"               #@param {type:"string"}
RUTA_REPORTES      = "Riesgo_y_Cumplimiento"


## 2.1 Configuración de archivos a cargar (parametrizable / escalable)
Cada entrada define: nombre del archivo CSV, tabla externa temporal (staging), tabla nativa destino y la columna(s) que trae el plano. Para agregar un 6° archivo en el futuro, **solo se agrega un diccionario a esta lista**.

In [ ]:
ARCHIVOS_CONFIG = [
    {
        "nombre_logico": "lista_mcc_gar_sbs",
        "archivo_csv": "Lista MCC GAR SBS.csv",
        "tabla_staging": "stg_lista_mcc_gar_sbs",
        "tabla_destino": "lista_mcc_gar_sbs",
        "columnas_origen": ["mcc_gar_sbs"],
        # Esquema EXPLICITO de la external table (evita que Omni infiera mal el esquema
        # y junte todo en una sola columna cuando el delimitador/quote no calzan)
        "columnas_tipos": [
            ("mcc_gar_sbs", "STRING"),
        ],
    },
    {
        "nombre_logico": "lista_restrictiva",
        "archivo_csv": "Lista Restrictiva.csv",
        "tabla_staging": "stg_lista_restrictiva",
        "tabla_destino": "lista_restrictiva",
        # El archivo nuevo viene con 22 columnas, separado por ',' (delimiter default, ya no ';').
        # IMPORTANTE: "columnas_tipos" debe declarar TODAS las columnas del CSV, EN EL MISMO ORDEN
        # FISICO en que vienen en el archivo (el match es por POSICION, no por nombre de encabezado).
        "columnas_tipos": [
            ("tipo_doc", "STRING"),
            ("dni", "STRING"),
            ("apellidos_nombres_rep_legal", "STRING"),
            ("ruc", "STRING"),
            ("nombre_comercial", "STRING"),
            ("cod_comercio", "STRING"),
            ("fecha_afiliacion", "STRING"),
            ("fecha_bloqueo", "STRING"),
            ("razon_social", "STRING"),
            ("mcc", "STRING"),
            ("actividad", "STRING"),
            ("producto", "STRING"),
            ("canal_afiliacion", "STRING"),
            ("direccion_comercial", "STRING"),
            ("distrito", "STRING"),
            ("telefono", "STRING"),
            ("telefono2", "STRING"),
            ("tipo_de_fraude", "STRING"),
            ("entidad_bancaria", "STRING"),
            ("cta_bancaria", "STRING"),
            ("origen_reporta", "STRING"),
            ("correo_electronico", "STRING"),
        ],
        # Subconjunto de columnas que efectivamente se inserta en la tabla nativa destino.
        # Si se necesitan mas columnas (ej. tipo_de_fraude, entidad_bancaria), agregarlas aqui
        # Y en el esquema de la tabla nativa "lista_restrictiva" en BigQuery.
        "columnas_origen": ["dni", "telefono", "telefono2", "cta_bancaria", "correo_electronico", "ruc"],
    },
    {
        "nombre_logico": "lista_pep",
        "archivo_csv": "Lista PEP.csv",
        "tabla_staging": "stg_lista_pep",
        "tabla_destino": "lista_pep",
        "columnas_origen": ["nrodocumento"],
        "columnas_tipos": [
            ("nrodocumento", "STRING"),
        ],
    },
    {
        "nombre_logico": "lista_pariente_pep",
        "archivo_csv": "Lista Pariente PEP.csv",
        "tabla_staging": "stg_lista_pariente_pep",
        "tabla_destino": "lista_pariente_pep",
        "columnas_origen": ["dnirel"],
        "columnas_tipos": [
            ("dnirel", "STRING"),
        ],
    },
    {
        "nombre_logico": "lista_zg_riesgo",
        "archivo_csv": "Lista ZG Riesgo.csv",
        "tabla_staging": "stg_lista_zg_riesgo",
        "tabla_destino": "lista_zg_riesgo",
        "columnas_origen": ["departamento"],
        "columnas_tipos": [
            ("departamento", "STRING"),
        ],
    },
]


# 3. Funciones reutilizables (módulo de carga)
Funciones genéricas e independientes entre sí (bajo acoplamiento). Cada una hace una sola cosa: crear la externa, construir el INSERT con auditoría, truncar+insertar, y limpiar.

In [ ]:
def crear_external_table_csv(tabla_staging: str, archivo_csv: str, columnas_tipos: list, field_delimiter: str = None) -> str:
    """Crea (o reemplaza) una External Table de BigQuery Omni apuntando al CSV en ADLS,
    declarando el ESQUEMA EXPLICITO (igual que el patron usado para Comercios 3DS).
    Esto evita que Omni infiera mal el esquema y junte todo en una sola columna cuando
    el delimitador o el quote no calzan con el archivo real.
    """
    uri = f'["{RUTA_BASE_ADLS}/{archivo_csv}"]'
    tabla_fqn = f"`{var_project_operation}.{var_dataset_stage}.{tabla_staging}`"
    delimiter = field_delimiter if field_delimiter else var_field_delimiter  # override por archivo si aplica

    esquema_sql = ",\n      ".join([f"{col} {tipo}" for col, tipo in columnas_tipos])

    sql = f"""
    CREATE OR REPLACE EXTERNAL TABLE {tabla_fqn}
    (
      {esquema_sql}
    )
    WITH CONNECTION `{var_connection}`
    OPTIONS (
      format = 'CSV',
      uris = {uri},
      field_delimiter = '{delimiter}',
      quote = '{var_quote}',
      skip_leading_rows = 1,
      allow_quoted_newlines = True,
      allow_jagged_rows = True
    );
    """
    clientBQ.query(sql).result()
    print(f"[OK] External table creada (esquema explicito, delimiter='{delimiter}'): {tabla_fqn}  <-  {archivo_csv}")
    return tabla_fqn


def materializar_tabla_temporal(tabla_staging_fqn: str, tabla_staging: str, columnas_origen: list) -> str:
    """IMPORTANTE: BigQuery Omni no permite escribir (CTAS/INSERT) dentro de la misma region
    Omni (azure-eastus2). Por eso, igual que en el procedimiento de AS400 (prc_load_as400_mcfv001),
    la externa (region Omni, var_dataset_stage) se materializa en una tabla NORMAL dentro de un
    dataset ESTANDAR de BigQuery (region US, var_dataset_tmp). Desde esa tabla normal si se puede
    insertar cross-project hacia var_project_storage.
    """
    tabla_tmp = f"tmp_{tabla_staging}"
    tabla_tmp_fqn = f"`{var_project_operation}.{var_dataset_tmp}.{tabla_tmp}`"  # dataset estandar, NO Omni
    cols_select = ",\n        ".join([f"TRIM({c}) AS {c}" for c in columnas_origen])

    sql = f"""
    CREATE OR REPLACE TABLE {tabla_tmp_fqn} AS
    SELECT
        {cols_select}
    FROM {tabla_staging_fqn}
    """
    clientBQ.query(sql).result()
    print(f"[OK] Tabla temporal materializada: {tabla_tmp_fqn}")
    return tabla_tmp_fqn


def truncar_tabla_destino(tabla_destino: str) -> str:
    """Trunca la tabla nativa destino antes de insertar el nuevo plano (regla de negocio)."""
    tabla_fqn = f"`{var_project_storage}.{var_dataset_destino}.{tabla_destino}`"
    sql = f"TRUNCATE TABLE {tabla_fqn};"
    clientBQ.query(sql).result()
    print(f"[OK] Tabla truncada: {tabla_fqn}")
    return tabla_fqn


def insertar_con_auditoria(tabla_tmp_fqn: str, tabla_destino_fqn: str, columnas_origen: list) -> None:
    """Inserta desde la tabla temporal (ya materializada, mismo truco que AS400) hacia la
    tabla nativa destino, agregando los campos de auditoria."""
    cols_destino = ", ".join(columnas_origen + ["process_date", "record_source", "load_date", "creation_user"])
    cols_select_tmp = ", ".join([f"a.{c}" for c in columnas_origen])

    sql = f"""
    INSERT INTO {tabla_destino_fqn} ({cols_destino})
    SELECT
        {cols_select_tmp},
        DATE('{VAR_PROCESS_DATE}')          AS process_date,
        '{RECORD_SOURCE}'                   AS record_source,
        DATETIME('{VAR_LOAD_DATE}')         AS load_date,
        '{CREATION_USER}'                   AS creation_user
    FROM {tabla_tmp_fqn} a
    """
    clientBQ.query(sql).result()
    print(f"[OK] Insert realizado en: {tabla_destino_fqn}")


def eliminar_tablas_temporales(tabla_staging_fqn: str, tabla_tmp_fqn: str) -> None:
    """Limpieza: elimina la external table y la tabla temporal materializada."""
    clientBQ.query(f"DROP TABLE IF EXISTS {tabla_staging_fqn};").result()
    clientBQ.query(f"DROP TABLE IF EXISTS {tabla_tmp_fqn};").result()
    print(f"[OK] Tablas temporales eliminadas: {tabla_staging_fqn} | {tabla_tmp_fqn}")


def procesar_archivo(config: dict) -> None:
    """Orquesta el flujo completo para UN archivo:
    crear externa (esquema explicito) -> materializar temporal -> truncar destino -> insertar -> limpiar.
    """
    print(f"\n--- Procesando: {config['nombre_logico']} ---")
    tabla_staging_fqn = crear_external_table_csv(
        config['tabla_staging'],
        config['archivo_csv'],
        config['columnas_tipos'],
        config.get('field_delimiter'),  # usa override si el archivo lo define, sino el default global
    )
    tabla_tmp_fqn      = materializar_tabla_temporal(tabla_staging_fqn, config['tabla_staging'], config['columnas_origen'])
    tabla_destino_fqn  = truncar_tabla_destino(config['tabla_destino'])
    insertar_con_auditoria(tabla_tmp_fqn, tabla_destino_fqn, config['columnas_origen'])
    eliminar_tablas_temporales(tabla_staging_fqn, tabla_tmp_fqn)


# 4. Ejecución de la carga (orquestación)
Itera sobre `ARCHIVOS_CONFIG`. Si falla un archivo, se reporta el error pero el resto continúa (mantenible / tolerante a fallos puntuales).

In [ ]:
errores = []
for cfg in ARCHIVOS_CONFIG:
    try:
        procesar_archivo(cfg)
    except Exception as e:
        print(f"[ERROR] Falló la carga de '{cfg['nombre_logico']}': {e}")
        errores.append((cfg['nombre_logico'], str(e)))

if errores:
    print("\n=== RESUMEN DE ERRORES ===")
    for nombre, err in errores:
        print(f" - {nombre}: {err}")
else:
    print("\n[OK] Todos los archivos fueron cargados correctamente.")


# 5. Reportes de Scoring de Riesgo
Cada reporte: ejecuta el query, lo materializa como tabla temporal (TTL gestionado por BigQuery con `CREATE TEMP TABLE` dentro de un script multi-statement), lo lleva a DataFrame y lo exporta a GCS en formato CSV.

In [ ]:
def ejecutar_query_a_dataframe(sql: str):
    """Ejecuta un SQL y retorna un DataFrame. Se apoya en TEMP TABLE dentro del propio script para
    aprovechar la sugerencia de negocio ('crear tabla temporal y después exportar')."""
    job = clientBQ.query(sql)
    return job.result().to_dataframe()


def exportar_dataframe_a_gcs(df, nombre_archivo: str) -> str:
    """Exporta un DataFrame como CSV COMPRIMIDO (.csv.gz) al bucket/ruta configurados."""
    bucket = client.bucket(bucket_name)

    csv_buffer = io.StringIO()
    df.to_csv(csv_buffer, sep=';', quoting=1, quotechar='"', index=False)

    # Comprimir el CSV en memoria (gzip) antes de subirlo
    gz_buffer = io.BytesIO()
    with gzip.GzipFile(fileobj=gz_buffer, mode='wb') as gz:
        gz.write(csv_buffer.getvalue().encode('utf-8'))
    gz_buffer.seek(0)

    blob_name = f"{RUTA_REPORTES}/{VAR_ANHO}/{VAR_MES}/{nombre_archivo}_{VAR_FECHA_ARCHIVO}.csv.gz"
    blob = bucket.blob(blob_name)
    blob.upload_from_file(gz_buffer, content_type='application/gzip')

    ruta_final = f"gs://{bucket_name}/{blob_name}"
    print(f"[OK] Reporte exportado (comprimido) a: {ruta_final}")
    return ruta_final


def generar_reporte(nombre_logico: str, nombre_archivo_salida: str, sql: str) -> None:
    print(f"\n--- Generando reporte: {nombre_logico} ---")
    df = ejecutar_query_a_dataframe(sql)
    print(f"[OK] Filas obtenidas: {len(df)}")
    exportar_dataframe_a_gcs(df, nombre_archivo_salida)


## 5.1 Reporte 1 — Scoring de Riesgos: Persona Natural

In [ ]:
SQL_REPORTE_1 = r"""
with cte_llaves as (
  select
    max(case when code = 'C_TELEPHONE'      then key      end) as tel_key,
    max(case when code = 'C_TELEPHONE'      then constant end) as tel_const,
    max(case when code = 'C_EMAIL'          then key      end) as email_key,
    max(case when code = 'C_EMAIL'          then constant end) as email_const,
    max(case when code = 'C_ACCOUNT_NUMBER' then key      end) as cta_key,
    max(case when code = 'C_ACCOUNT_NUMBER' then constant end) as cta_const
  from `prd-izipay-data-sensitive.secure_secrets.config_protected_data`
  where code in ('C_TELEPHONE', 'C_EMAIL', 'C_ACCOUNT_NUMBER')
),
base_comercio_filtrada as (
  -- CTE base sobre m_comercio: aqui se aplican los filtros de parque (CA4)
  -- y se agregan los campos dealer / tipcom (CA3)
  select
    a.cod_comercio,
    a.party_id_izi_representante,
    a.party_id_izi,
    a.cod_situacion_comercio,
    a.desc_situacion_comercio,
    a.fecha_apertura_comercio,
    a.cod_giro_comercio                                                                              as mcc,
    a.razon_social_dealer                                                                             as dealer,   -- CA3: campo nuevo
    a.nom_producto                                                                                     as tipcom,   -- CA3: campo nuevo
    upper(trim(a.departamento_comercio))                                                             as departamento_comercio_clean,
    a.subtipo_documento_facilitador,
    upper(trim(AEAD.DECRYPT_STRING(k.tel_key,   a.telefono_comercio,          k.tel_const)))        as telefono_comercio_dec,
    upper(trim(AEAD.DECRYPT_STRING(k.email_key, a.correo_representante_legal, k.email_const)))      as correo_representante_legal_dec,
    upper(trim(AEAD.DECRYPT_STRING(k.cta_key,   a.num_cuenta_comercio,        k.cta_const)))        as num_cuenta_comercio_dec,
    segmento_parque
  from `prd-izipay-data-storage-pv.master_party.m_comercio` a
  cross join cte_llaves k
  where a.compania in ('PMP', 'IZIPAY')
    and a.flag_parque = true
    and a.subtipo_documento_facilitador in ('CE', 'DNI', 'RUC 10', 'RUC 15', 'RUC 17')
    and a.cod_situacion_comercio not in ('3', '9')                                                                    -- CA4: filtro nuevo
    and a.nom_producto not in ('CAJERO CORRESPONSAL', 'INTEROPERABILIDAD VISANET', 'VENDEMAS', 'IZIPAY YA')          -- CA4: filtro nuevo
),
aux_iden_party_data_control as (
  select
    party_id_izi,
    document_number,
    trim(document_number) as document_number_clean
  from `prd-izipay-data-sensitive.master_pii.iden_party_data_control`
  qualify row_number() over (partition by party_id_izi order by document_number desc) = 1
),
aux_volumen_historico as (
  select
    trim(t.pdcest) as cod_comercio,
    sum(t.importe) as volumen_trx_historico
  from `prd-izipay-data-storage-pv.master_transaction.t_detalle_transacciones` t
  inner join base_comercio_filtrada b on (t.pdcest = b.cod_comercio)
  where t.process_date >= b.fecha_apertura_comercio
  and t.process_date >= date_sub(current_date('America/Lima'), interval 10 year)
  group by 1
),
aux_volumen_12m as (
  select
   trim(t.pdcest) as cod_comercio,
   sum(t.importe) as volumen_total_trx_ult_12_m
  from `prd-izipay-data-storage-pv.master_transaction.t_detalle_transacciones` t
  inner join base_comercio_filtrada b on t.pdcest = b.cod_comercio
  where t.process_date >= date_sub(current_date('America/Lima'), interval 12 month)
  group by 1
),
aux_lista_restrictiva_dni as (
  select distinct
   trim(dni) as dni
  from  `prd-izipay-data-storage-pv.raw_dataentry_riesgos.lista_restrictiva`
  where dni is not null
),
aux_lista_restrictiva_tel as (
  select distinct trim(telefono) as telefono
  from `prd-izipay-data-storage-pv.raw_dataentry_riesgos.lista_restrictiva`
  where trim(telefono) is not null

  union distinct

  select distinct trim(telefono2) as telefono
  from `prd-izipay-data-storage-pv.raw_dataentry_riesgos.lista_restrictiva`
  where trim(telefono2) is not null
),
aux_lista_restrictiva_cta as (
  select distinct
   trim(cta_bancaria) as cta_bancaria
  from `prd-izipay-data-storage-pv.raw_dataentry_riesgos.lista_restrictiva`
  where cta_bancaria is not null
),
aux_lista_restrictiva_correo as (
  select distinct
   upper(trim(correo_electronico)) as correo_electronico
  from `prd-izipay-data-storage-pv.raw_dataentry_riesgos.lista_restrictiva`
  where correo_electronico is not null
),
aux_lista_pep as (
  select distinct
   trim(nrodocumento) as nrodocumento
  from `prd-izipay-data-storage-pv.raw_dataentry_riesgos.lista_pep`
  where nrodocumento is not null
),
aux_lista_pariente_pep as (
  select distinct
    trim(dnirel) as dnirel
  from `prd-izipay-data-storage-pv.raw_dataentry_riesgos.lista_pariente_pep`
  where dnirel is not null
),
aux_lista_zg_riesgo as (
  select distinct
    upper(trim(departamento)) as departamento
  from `prd-izipay-data-storage-pv.raw_dataentry_riesgos.lista_zg_riesgo`
  where departamento is not null
),
aux_lista_mcc_gar_sbs as (
  select distinct
    trim(mcc_gar_sbs) as mcc_gar_sbs
  from `prd-izipay-data-storage-pv.raw_dataentry_riesgos.lista_mcc_gar_sbs`
  where mcc_gar_sbs is not null
),
aux_base_comercio_filtrada as (
select
  a.cod_comercio,
  a.cod_situacion_comercio,
  a.desc_situacion_comercio,
  a.fecha_apertura_comercio,
  coalesce(b.volumen_trx_historico,      0)                                  as volumen_trx_historico,
  coalesce(b2.volumen_total_trx_ult_12_m, 0)                                  as volumen_total_trx_ult_12_m,
  a.mcc,
  a.dealer,                                                                                                          -- CA3: campo nuevo (passthrough)
  a.tipcom,                                                                                                          -- CA3: campo nuevo (passthrough)
  c.document_number                                                           as nro_documento_repres_legal,
  c1.document_number                                                          as nro_documento_afiliacion,
  a.telefono_comercio_dec                                                     as telefono_comercio,
  a.correo_representante_legal_dec                                            as correo_representante_legal,
  a.num_cuenta_comercio_dec                                                   as nrocta,
  a.departamento_comercio_clean                                               as departamento_comercio,
  a.subtipo_documento_facilitador,
  case when lr_dni.DNI                 is not null then 1 else 0 end          as nro_doc_de_identidad,
  case
    when a.subtipo_documento_facilitador = 'CE'                                then 1
    when a.subtipo_documento_facilitador in ('DNI','RUC 10','RUC 15','RUC 17') then 0
    else null
  end                                                                         as nacionalidad,
  case when lr_tel.TELEFONO            is not null then 1 else 0 end          as telefono,
  case when lr_correo.CORREO_ELECTRONICO is not null then 1 else 0 end        as correo_electronico,
  case when pep.NRODOCUMENTO           is not null then 1 else 0 end          as pep,
  case when ppep.DNIREL                is not null then 1 else 0 end          as pariente_pep,
  case when zg.departamento            is not null then 1 else 0 end          as departamento,
  case when lr_cta.CTA_BANCARIA        is not null then 1 else 0 end          as nro_cuenta,
  case when mcc_gar.mcc_gar_sbs        is not null then 1 else 0 end          as mcc_gar_sbs,
  segmento_parque
from base_comercio_filtrada a
left join aux_volumen_historico b  on (a.cod_comercio = b.cod_comercio)
left join aux_volumen_12m b2  on (a.cod_comercio = b2.cod_comercio)
left join  aux_iden_party_data_control    c        on c.party_id_izi              = a.party_id_izi_representante
left join  aux_iden_party_data_control    c1       on (c1.party_id_izi = a.party_id_izi )
left join  aux_lista_restrictiva_dni      lr_dni   on c.document_number_clean      = lr_dni.DNI
left join  aux_lista_restrictiva_tel      lr_tel   on a.telefono_comercio_dec      = lr_tel.TELEFONO
left join  aux_lista_restrictiva_correo   lr_correo on a.correo_representante_legal_dec = lr_correo.CORREO_ELECTRONICO
left join  aux_lista_restrictiva_cta      lr_cta   on a.num_cuenta_comercio_dec    = lr_cta.CTA_BANCARIA
left join  aux_lista_pep                  pep      on c.document_number_clean      = pep.NRODOCUMENTO
left join  aux_lista_pariente_pep         ppep     on c.document_number_clean      = ppep.DNIREL
left join  aux_lista_zg_riesgo            zg       on a.departamento_comercio_clean = zg.departamento
left join  aux_lista_mcc_gar_sbs          mcc_gar  on trim(a.mcc)                  = mcc_gar.mcc_gar_sbs
),
aux_base_comercio_filtrada_puntaje as (
select
 cod_comercio,
 cod_situacion_comercio,
 desc_situacion_comercio,
 fecha_apertura_comercio,
 volumen_trx_historico,
 volumen_total_trx_ult_12_m,
 mcc,
 segmento_parque,
 dealer,                                                                      -- CA3: campo nuevo (passthrough)
 tipcom,                                                                      -- CA3: campo nuevo (passthrough)
 nro_documento_repres_legal,
 nro_documento_afiliacion,
 telefono_comercio,
 REGEXP_REPLACE(correo_representante_legal, r'[\r\n]+', '') as correo_representante_legal,
 nrocta,
 departamento_comercio,
 subtipo_documento_facilitador,
 nro_doc_de_identidad,
 nacionalidad,
 telefono,
 correo_electronico,
 pep,
 pariente_pep,
 mcc_gar_sbs,
 departamento,
 nro_cuenta,
 case when nro_doc_de_identidad = 1 then 6.0 else 0.0 end as p1,
 case when nacionalidad = 1 then 0.5 else 0.0 end as p2,
 case when telefono = 1 then 6.0 else 0.0 end as p3,
 case when correo_electronico = 1 then 6.0 else 0.0 end as p4,
 case when pep = 1 then 2.0 else 0.0 end as p5,
 case when pariente_pep = 1 then 2.0 else 0.0 end as p6,
 case when mcc_gar_sbs = 1 then 6.0 else 0.0 end as p7,
 case when departamento = 1 then 0.5 else 0.0 end as p8,
 case when nro_cuenta = 1 then 6.0 else 0.0 end as p9
from aux_base_comercio_filtrada
)
select
 cod_comercio                as comercio
,nro_documento_afiliacion    as documento_afiliacion                                                                 -- CA3: campo nuevo
,desc_situacion_comercio     as situacion
,fecha_apertura_comercio     as fecha_de_afiliacion
,mcc                         as mcc
,segmento_parque             as segmento
,dealer                      as dealer                                                                               -- CA3: campo nuevo
,tipcom                      as tipcom                                                                               -- CA3: campo nuevo
,volumen_trx_historico
,volumen_total_trx_ult_12_m
,nro_doc_de_identidad
,nacionalidad
,telefono
,correo_electronico
,pep
,pariente_pep
,mcc_gar_sbs
,departamento
,nro_cuenta
,p1
,p2
,p3
,p4
,p5
,p6
,p7
,p8
,p9
,(p1 + p2 + p3 + p4 + p5 + p6 + p7 + p8 + p9) as puntaje_total
,case
    when p1 + p2 + p3 + p4 + p5 + p6 + p7 + p8 + p9 = 0 then 'Bajo'
    when p1 + p2 + p3 + p4 + p5 + p6 + p7 + p8 + p9 between 0.5 and 1 then 'Moderado'
    when p1 + p2 + p3 + p4 + p5 + p6 + p7 + p8 + p9 between 2 and 5 then 'Alto'
    when p1 + p2 + p3 + p4 + p5 + p6 + p7 + p8 + p9 >= 6 then 'Extremo'
 end as nivel_riesgo
from aux_base_comercio_filtrada_puntaje
"""

generar_reporte("Scoring Persona Natural", "Scoring_de_Riesgos_Persona_Natural", SQL_REPORTE_1)


## 5.2 Reporte 2 — Scoring de Riesgos: Persona Jurídica

In [ ]:
SQL_REPORTE_2 = r"""
with cte_llaves as (
  select
    max(case when code = 'C_TELEPHONE'      then key      end) as tel_key,
    max(case when code = 'C_TELEPHONE'      then constant end) as tel_const,
    max(case when code = 'C_EMAIL'          then key      end) as email_key,
    max(case when code = 'C_EMAIL'          then constant end) as email_const,
    max(case when code = 'C_ACCOUNT_NUMBER' then key      end) as cta_key,
    max(case when code = 'C_ACCOUNT_NUMBER' then constant end) as cta_const
  from `prd-izipay-data-sensitive.secure_secrets.config_protected_data`
  where code in ('C_TELEPHONE', 'C_EMAIL', 'C_ACCOUNT_NUMBER')
),
base_comercio_filtrada as (
  -- CTE base sobre m_comercio: aqui se aplican los filtros de parque (CA4)
  -- y se agregan los campos dealer / tipcom (CA3)
  select
    a.cod_comercio,
    a.party_id_izi,
    a.party_id_izi_representante,
    a.cod_situacion_comercio,
    a.desc_situacion_comercio,
    a.fecha_apertura_comercio,
    a.cod_giro_comercio                                                                              as mcc,
    a.razon_social_dealer                                                                             as dealer,   -- CA3: campo nuevo
    a.nom_producto                                                                                     as tipcom,   -- CA3: campo nuevo
    upper(trim(a.departamento_comercio))                                                             as departamento_comercio_clean,
    a.subtipo_documento_facilitador,
    upper(trim(AEAD.DECRYPT_STRING(k.tel_key,   a.telefono_comercio,          k.tel_const)))        as telefono_comercio_dec,
    upper(trim(AEAD.DECRYPT_STRING(k.email_key, a.correo_representante_legal, k.email_const)))      as correo_representante_legal_dec,
    upper(trim(AEAD.DECRYPT_STRING(k.cta_key,   a.num_cuenta_comercio,        k.cta_const)))        as num_cuenta_comercio_dec,
    segmento_parque
  from `prd-izipay-data-storage-pv.master_party.m_comercio` a
  cross join cte_llaves k
  where a.compania in ('PMP', 'IZIPAY')
    and a.flag_parque = true
    and a.subtipo_documento_facilitador in ('RUC 20')
    and a.cod_situacion_comercio not in ('3', '9')                                                                    -- CA4: filtro nuevo
    and a.nom_producto not in ('CAJERO CORRESPONSAL', 'INTEROPERABILIDAD VISANET', 'VENDEMAS', 'IZIPAY YA')          -- CA4: filtro nuevo
),
aux_iden_party_data_control as (
  select
    party_id_izi,
    document_number
  from `prd-izipay-data-sensitive.master_pii.iden_party_data_control`
  qualify row_number() over (partition by party_id_izi order by document_number desc) = 1
),
aux_volumen_historico as (
 select
    trim(t.pdcest) as cod_comercio,
    sum(t.importe) as volumen_trx_historico
  from `prd-izipay-data-storage-pv.master_transaction.t_detalle_transacciones` t
  inner join base_comercio_filtrada b on (t.pdcest = b.cod_comercio)
  where t.process_date >= b.fecha_apertura_comercio
  and t.process_date >= date_sub(current_date('America/Lima'), interval 10 year)
  group by 1
),
aux_volumen_12m as (
  select
    trim(t.pdcest) as cod_comercio,
    sum(t.importe) as volumen_total_trx_ult_12_m
  from `prd-izipay-data-storage-pv.master_transaction.t_detalle_transacciones` t
  inner join base_comercio_filtrada b
    on t.pdcest = b.cod_comercio
  where t.process_date >= date_sub(current_date('America/Lima'), interval 12 month)
  group by 1
),
aux_lista_restrictiva_dni as (
 select distinct
  trim(dni) as dni
 from `prd-izipay-data-storage-pv.raw_dataentry_riesgos.lista_restrictiva`
 where dni is not null
),
aux_lista_restrictiva_tel as (
 select distinct
  trim(telefono) as telefono
 from `prd-izipay-data-storage-pv.raw_dataentry_riesgos.lista_restrictiva`
 where trim(telefono) is not null

 union distinct

 select distinct
   trim(telefono2) as telefono
 from `prd-izipay-data-storage-pv.raw_dataentry_riesgos.lista_restrictiva`
 where trim(telefono2) is not null
),
aux_lista_restrictiva_cta as (
 select distinct
  trim(cta_bancaria) as cta_bancaria
 from `prd-izipay-data-storage-pv.raw_dataentry_riesgos.lista_restrictiva`
 where cta_bancaria is not null
),
aux_lista_restrictiva_correo as (
 select distinct
  upper(trim(correo_electronico)) as correo_electronico
 from `prd-izipay-data-storage-pv.raw_dataentry_riesgos.lista_restrictiva`
 where correo_electronico is not null
),
aux_lista_restrictiva_ruc as (
 select distinct
  trim(ruc) as ruc
 from `prd-izipay-data-storage-pv.raw_dataentry_riesgos.lista_restrictiva`
 where ruc is not null
),
aux_lista_pep as (
  select distinct
   trim(nrodocumento) as nrodocumento
  from `prd-izipay-data-storage-pv.raw_dataentry_riesgos.lista_pep`
  where nrodocumento is not null
),
aux_lista_pariente_pep as (
  select distinct
    trim(dnirel) as dnirel
  from `prd-izipay-data-storage-pv.raw_dataentry_riesgos.lista_pariente_pep`
  where dnirel is not null
),
aux_lista_zg_riesgo as (
  select distinct
    upper(trim(departamento)) as departamento
  from `prd-izipay-data-storage-pv.raw_dataentry_riesgos.lista_zg_riesgo`
  where departamento is not null
),
aux_lista_mcc_gar_sbs as (
  select distinct
    trim(mcc_gar_sbs) as mcc_gar_sbs
  from `prd-izipay-data-storage-pv.raw_dataentry_riesgos.lista_mcc_gar_sbs`
  where mcc_gar_sbs is not null
),
aux_base_comercio_juridico_filtrada as (
select
  a.cod_comercio,
  a.cod_situacion_comercio,
  a.desc_situacion_comercio,
  a.fecha_apertura_comercio,
  coalesce(b.volumen_trx_historico,      0)                                  as volumen_trx_historico,
  coalesce(b2.volumen_total_trx_ult_12_m, 0)                                 as volumen_total_trx_ult_12_m,
  a.mcc,
  a.dealer,                                                                                                          -- CA3: campo nuevo (passthrough)
  a.tipcom,                                                                                                          -- CA3: campo nuevo (passthrough)
  c.document_number                                                           as nro_documento_ruc_afiliacion,
  d.document_number                                                           as nro_documento_representante_legal,
  a.telefono_comercio_dec                                                     as telefono_comercio,
  a.correo_representante_legal_dec                                            as correo_representante_legal,
  a.num_cuenta_comercio_dec                                                   as nrocta,
  a.departamento_comercio_clean                                               as departamento_comercio,
  a.subtipo_documento_facilitador,
  case when lr_dni.DNI                 is not null then 1 else 0 end          as nro_doc_rl,
  case when lr_ruc.RUC                 is not null then 1 else 0 end          as nro_doc_ruc,
  0                                                                           as nacionalidad,
  case when lr_tel.TELEFONO            is not null then 1 else 0 end          as telefono,
  case when lr_correo.CORREO_ELECTRONICO is not null then 1 else 0 end        as correo_electronico,
  case when pep.NRODOCUMENTO           is not null then 1 else 0 end          as pep,
  case when ppep.DNIREL                is not null then 1 else 0 end          as pariente_pep,
  case when zg.departamento            is not null then 1 else 0 end          as departamento,
  case when lr_cta.CTA_BANCARIA        is not null then 1 else 0 end          as nro_cuenta,
  case when mcc_gar.mcc_gar_sbs        is not null then 1 else 0 end          as mcc_gar_sbs,
  segmento_parque
from base_comercio_filtrada a
left join aux_volumen_historico b  on ( b.cod_comercio = a.cod_comercio)
left join aux_volumen_12m b2  on ( b2.cod_comercio = a.cod_comercio )
left join  aux_iden_party_data_control    c        on c.party_id_izi               = a.party_id_izi
left join  aux_iden_party_data_control    d        on d.party_id_izi               = a.party_id_izi_representante
left join  aux_lista_restrictiva_dni      lr_dni   on d.document_number      = lr_dni.DNI
left join  aux_lista_restrictiva_ruc      lr_ruc   on c.document_number      = lr_ruc.ruc
left join  aux_lista_restrictiva_tel      lr_tel   on a.telefono_comercio_dec      = lr_tel.TELEFONO
left join  aux_lista_restrictiva_correo   lr_correo on a.correo_representante_legal_dec = lr_correo.CORREO_ELECTRONICO
left join  aux_lista_restrictiva_cta      lr_cta   on a.num_cuenta_comercio_dec    = lr_cta.CTA_BANCARIA
left join  aux_lista_pep                  pep      on d.document_number      = pep.NRODOCUMENTO
left join  aux_lista_pariente_pep         ppep     on d.document_number      = ppep.DNIREL
left join  aux_lista_zg_riesgo            zg       on a.departamento_comercio_clean = zg.departamento
left join  aux_lista_mcc_gar_sbs          mcc_gar  on trim(a.mcc)                  = mcc_gar.mcc_gar_sbs
),
aux_base_puntaje as (
select
    cod_comercio,
    cod_situacion_comercio,
    desc_situacion_comercio,
    fecha_apertura_comercio,
    volumen_trx_historico,
    volumen_total_trx_ult_12_m,
    mcc,
    segmento_parque,
    dealer,                                                                   -- CA3: campo nuevo (passthrough)
    tipcom,                                                                   -- CA3: campo nuevo (passthrough)
    nro_documento_ruc_afiliacion,
    nro_documento_representante_legal,
    telefono_comercio,
    correo_representante_legal,
    nrocta,
    departamento_comercio,
    subtipo_documento_facilitador,
    nro_doc_ruc as nro_ruc,
    nro_doc_rl as nro_doc_rl,
    nacionalidad,
    telefono,
    correo_electronico,
    pep,
    pariente_pep,
    mcc_gar_sbs,
    departamento,
    nro_cuenta,
    case when nro_doc_ruc = 1 then 6.0 else 0.0 end as p1,
    case when nro_doc_rl = 1 then 6.0 else 0.0 end as p2,
    case when nacionalidad = 1 then 0.5 else 0.0 end as p3,
    case when telefono = 1 then 6.0 else 0.0 end as p4,
    case when correo_electronico = 1 then 6.0 else 0.0 end as p5,
    case when pep = 1 then 2.0 else 0.0 end as p6,
    case when pariente_pep = 1 then 2.0 else 0.0 end as p7,
    case when mcc_gar_sbs = 1 then 6.0 else 0.0 end as p8,
    case when departamento = 1 then 0.5 else 0.0 end as p9,
    case when nro_cuenta = 1 then 6.0 else 0.0 end as p10
from aux_base_comercio_juridico_filtrada
)
select
 cod_comercio                  as comercio
,nro_documento_ruc_afiliacion  as documento_afiliacion                                                                -- CA3: campo nuevo
,desc_situacion_comercio       as situacion
,fecha_apertura_comercio       as fecha_de_afiliacion
,mcc                           as mcc
,segmento_parque               as segmento
,dealer                        as dealer                                                                              -- CA3: campo nuevo
,tipcom                        as tipcom                                                                              -- CA3: campo nuevo
,volumen_trx_historico         as volumen_trx_historico
,volumen_total_trx_ult_12_m    as volumen_total_trx_ult_12_m
,nro_ruc                       as nro_ruc
,nro_doc_rl                    as nro_doc_rl
,nacionalidad                  as nacionalidad
,telefono                      as telefono
,correo_electronico            as correo_electronico
,pep                           as pep
,pariente_pep                  as pariente_pep
,mcc_gar_sbs                   as mcc_gar_sbs
,departamento                  as departamento
,nro_cuenta                    as nro_cuenta
,p1, p2, p3, p4, p5, p6, p7, p8, p9, p10
,p1 + p2 + p3 + p4 + p5 + p6 + p7 + p8 + p9 + p10 as puntaje_total
,case
    when p1 + p2 + p3 + p4 + p5 + p6 + p7 + p8 + p9 + p10 = 0  then 'Bajo'
    when p1 + p2 + p3 + p4 + p5 + p6 + p7 + p8 + p9 + p10      between 0.5 and 1 then 'Moderado'
    when p1 + p2 + p3 + p4 + p5 + p6 + p7 + p8 + p9 + p10      between 2 and 5 then 'Alto'
    when p1 + p2 + p3 + p4 + p5 + p6 + p7 + p8 + p9 + p10 >= 6 then 'Extremo'
  end as nivel_riesgo
from aux_base_puntaje
"""

generar_reporte("Scoring Persona Jurídica", "Scoring_de_Riesgos_Persona_Juridica", SQL_REPORTE_2)


# 6. Notas de diseño

- **Parametrizable**: fechas, proyectos, dataset, conexión Omni, bucket y rutas están en variables `#@param` en la sección 2; nada está *hardcodeado* dentro de las funciones.
- **Flexible/Escalable**: para una 6ta lista, se agrega un diccionario en `ARCHIVOS_CONFIG`; el loop de la sección 4 la procesa sin cambios de código.
- **Modular**: cada función (`crear_external_table_csv`, `truncar_tabla_destino`, `insertar_con_auditoria`, `eliminar_external_table`, `ejecutar_query_a_dataframe`, `exportar_dataframe_a_gcs`) hace una sola cosa y puede reutilizarse en otros notebooks.
- **Desacoplado**: la lógica de carga de listas no depende de la lógica de reportes; se podrían correr en notebooks/DAGs separados sin tocar código.
- **Mantenible**: errores por archivo no detienen todo el proceso; quedan registrados en `errores` para revisión.
- **Pendiente de confirmar con el equipo de Data Engineering**: nombre exacto del `var_connection`/dataset Omni habilitado para esta ruta de ADLS (Scoring), y si los CSV traen encabezado (`skip_leading_rows=1`) y delimitador `,` — ajustar si es `;`.